# Packages

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.recursive import ForecasterRecursive
from sklearn.neighbors import KNeighborsRegressor
from skforecast.model_selection import bayesian_search_forecaster
from skforecast.model_selection._split import TimeSeriesFold
from tslearn.neighbors import KNeighborsTimeSeriesRegressor
import numpy as np
import kagglehub

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
%load_ext nb_black

# Data

In [ ]:
# Download the dataset from Kaggle
path = kagglehub.dataset_download("sumanthvrao/daily-climate-time-series-data")
print("Path to dataset files:", path)

In [ ]:
data_train =\
pd.read_csv('/Users/egorhowell/.cache/kagglehub/datasets/sumanthvrao/daily-climate-time-series-data/versions/3/DailyDelhiClimateTrain.csv')

In [ ]:
data_test =\
pd.read_csv('/Users/egorhowell/.cache/kagglehub/datasets/sumanthvrao/daily-climate-time-series-data/versions/3/DailyDelhiClimateTest.csv')

In [ ]:
data_train

In [ ]:
data_test

In [ ]:
# Overlap in training data, so remove last entry
data_train = data_train[:-1].copy()

In [ ]:
data_train

# EDA

In [ ]:
# Ensure datetime type
data_train['date'] = pd.to_datetime(data_train['date'], errors='coerce')
data_test['date']  = pd.to_datetime(data_test['date'], errors='coerce')

In [ ]:
# Set dataset name for combination later
data_train.loc[:, 'dataset'] = 'train'
data_test.loc[:, 'dataset'] = 'test'

In [ ]:
# Combine data for easier plotting later
data_combined = pd.concat([data_train, data_test], ignore_index=True)
data_combined['date'] = pd.to_datetime(data_combined['date'])

In [ ]:
# Some basic EDA
print(data_train.dtypes)
print(data_train.isnull().sum())
print(data_train.describe())

In [ ]:
# Plot our data through time
variables = ['meantemp', 'humidity', 'wind_speed', 'meanpressure']

for var in variables:
    fig = px.line(
        data_combined,
        x='date',
        y=var,
        color='dataset',
        title=f"{var.capitalize()} Over Time",
        labels={'date': 'Date', var: var.capitalize()},
        color_discrete_map={'train': 'blue', 'test': 'orange'},
        template="simple_white"
    )
    fig.show()

# K-Nearest-Neighbour

In [ ]:
# Initialise our model
knn_model = ForecasterRecursive(regressor=KNeighborsRegressor(n_neighbors=5), lags=365)

In [ ]:
# Fit the model
knn_model.fit(y=data_train['meantemp'])

In [ ]:
# Forecast our data
forecast_knn = knn_model.predict(steps=len(data_test))

In [ ]:
mae = mean_absolute_error(data_test['meantemp'], forecast_knn)
rmse = np.sqrt(mean_squared_error(data_test['meantemp'], forecast_knn))

print(f"\nMAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

In [ ]:
# Plot the forecast along with the real data we observe
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=data_test['date'],
    y=forecast_knn,
    mode='lines',
    name='Forecast KNN'
))


fig.add_trace(go.Scatter(
    x=data_test['date'],
    y=data_test['meantemp'],
    mode='lines',
    name='Test',
))

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Mean Temperature (°C)',
    legend_title='Legend',
    template="simple_white",
    width=900,
    height=500
)

fig.show()


# KNN Better

## Hyperparameter Tuning

In [ ]:
# Initialise our model
knn_model_tune = ForecasterRecursive(regressor=KNeighborsRegressor(n_neighbors=3), lags=15)

In [ ]:
# Hyperparameters search space
def search_space(trial):
    search_space  = {
        'lags' : trial.suggest_int('lags', 2, 600),
        'n_neighbors' : trial.suggest_int('n_neighbors', 2, 100),
    } 
    return search_space

In [ ]:
# Our cross validation object
cv = TimeSeriesFold(steps=len(data_test), initial_train_size=int(len(data_train) * 0.8))

In [ ]:
# Perform Bayesian optimisation
best_params, best_metric = bayesian_search_forecaster(
    forecaster=knn_model_tune,
    cv=cv,
    y=data_train['meantemp'],
    search_space=search_space,
    metric='mean_absolute_error',
    n_trials=200,
    verbose=True
)

In [ ]:
best_params

In [ ]:
# Extract best lags and n_neighbors
best_lags = best_params['lags'].iloc[0]    
best_n_neighbors = best_params['params'].iloc[0]['n_neighbors']

print("Best lags:", best_lags)
print("Best n_neighbors:", best_n_neighbors)

## Fit Best Model

In [ ]:
# Initialise our model
knn_model_best = ForecasterRecursive(regressor=KNeighborsRegressor(n_neighbors=best_n_neighbors), lags=len(best_lags))

In [ ]:
# Fit the model
knn_model_best.fit(y=data_train['meantemp'])

In [ ]:
# Forecast our data
forecast_knn_best = knn_model_best.predict(steps=len(data_test))

In [ ]:
mae = mean_absolute_error(data_test['meantemp'], forecast_knn_best)
rmse = np.sqrt(mean_squared_error(data_test['meantemp'], forecast_knn_best))

print(f"\nMAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

In [ ]:
# Plot the forecast along with the real data we observe
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=data_test['date'],
    y=forecast_knn,
    mode='lines',
    name='Forecast KNN'
))

fig.add_trace(go.Scatter(
    x=data_test['date'],
    y=forecast_knn_best,
    mode='lines',
    name='Forecast KNN Better'
))

fig.add_trace(go.Scatter(
    x=data_test['date'],
    y=data_test['meantemp'],
    mode='lines',
    name='Test',
))

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Mean Temperature (°C)',
    legend_title='Legend',
    template="simple_white",
    width=900,
    height=500
)

fig.show()


# KNN with DTW

In [ ]:
# Convert to vector format for DTW
series = data_train['meantemp'].values.reshape(-1, 1)

In [ ]:
# Create sliding window for input
def create_dataset(series, m=15):
    X, y = [], []
    for i in range(len(series) - m):
        X.append(series[i:i+m])
        y.append(series[i+m])
    return np.array(X), np.array(y)

X, y = create_dataset(series, len(best_lags))

In [ ]:
X[0].shape

In [ ]:
# Fit model with distance metric DTW
knn_dtw = KNeighborsTimeSeriesRegressor(n_neighbors=best_n_neighbors, metric="dtw")
knn_dtw.fit(X, y)

In [ ]:
# Initialise last window for recursive forecasting
last_window = series[-len(best_lags):]
window = last_window.copy()

In [ ]:
# Forecast horizon and list to forecasts
n_steps = len(data_test)
forecasts = []

In [ ]:
for _ in range(n_steps):
    pred = knn_dtw.predict(window.reshape(1, len(best_lags), 1))
    pred_scalar = pred.item()

    forecasts.append(pred_scalar)

    # New window, so remove first current element
    window = np.vstack([window[1:], [[pred_scalar]]])


In [ ]:
# Convert predictions list to numpy array
forecast_dtw = np.array(forecasts)

In [ ]:
mae = mean_absolute_error(data_test['meantemp'], forecast_dtw)
rmse = np.sqrt(mean_squared_error(data_test['meantemp'], forecast_dtw))

print(f"\nMAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

In [ ]:
# Plot the forecast along with the real data we observe
fig = go.Figure()


fig.add_trace(go.Scatter(
    x=data_test['date'],
    y=forecast_knn,
    mode='lines',
    name='Forecast KNN'
))

fig.add_trace(go.Scatter(
    x=data_test['date'],
    y=forecast_knn_best,
    mode='lines',
    name='Forecast KNN Better'
))

fig.add_trace(go.Scatter(
    x=data_test['date'],
    y=forecast_dtw,
    mode='lines',
    name='Forecast DTW'
))

fig.add_trace(go.Scatter(
    x=data_test['date'],
    y=data_test['meantemp'],
    mode='lines',
    name='Test',
))

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Mean Temperature (°C)',
    legend_title='Legend',
    template="simple_white",
    width=900,
    height=500
)

fig.show()
